In [4]:
# NB03b_data_dictionary_check.ipynb
# Study 2: Verify variable definitions and detect duplicate/redundant features
#          before association rule mining (NB04)
#
# Purpose:
#   ebit_to_assets and Attr14 showed VIF ~ 10^12 in NB03, indicating near-perfect
#   linear dependence. This notebook identifies which Attr columns are aliases
#   of named variables and flags remaining duplicates for exclusion in NB04.
#
# Input  : data/processed/upgrade_cohort.parquet
#           results/tables/NB03_02_top_candidates.csv
# Output : results/tables/NB03b_variable_map.csv  — final deduplicated variable list

import os
import numpy as np
import pandas as pd

PROC_DIR  = "../data/processed/"
TABLE_DIR = "../results/tables/"

cohort     = pd.read_parquet(PROC_DIR + "upgrade_cohort.parquet")
candidates = pd.read_csv(TABLE_DIR + "NB03_02_top_candidates.csv", index_col=0)

print(f"Candidate variables from NB03 : {len(candidates)}")
print(candidates.index.tolist())


# ── Known variable name map (from UCI dataset documentation) ──────────────────
# Source: Zięba et al. (2016), Polish Companies Bankruptcy dataset
# https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy+data

ATTR_MAP = {
    "Attr1" : "net profit / total assets",
    "Attr2" : "total liabilities / total assets",
    "Attr3" : "working capital / total assets",
    "Attr4" : "current assets / short-term liabilities",
    "Attr5" : "cash / current liabilities",
    "Attr6" : "retained earnings / total assets",
    "Attr7" : "EBIT / total assets",
    "Attr8" : "book value of equity / total liabilities",
    "Attr9" : "sales / total assets",
    "Attr10": "equity / total assets",
    "Attr11": "gross profit + extraordinary items + financial expenses / total assets",
    "Attr12": "gross profit / short-term liabilities",
    "Attr13": "gross profit + depreciation / sales",
    "Attr14": "gross profit + interest / total assets",
    "Attr15": "total liabilities × 365 / (gross profit + depreciation)",
    "Attr16": "gross profit / total assets",
    "Attr17": "total assets / total liabilities",
    "Attr18": "gross profit / total assets (alt)",
    "Attr19": "gross profit / sales",
    "Attr20": "inventory × 365 / sales",
    "Attr21": "sales (t) / sales (t-1)",
    "Attr22": "profit on operating activities / total assets",
    "Attr23": "net profit / sales",
    "Attr24": "gross profit (3-year avg) / total assets",
    "Attr25": "equity / total assets (alt)",
    "Attr26": "gross profit / (depreciation × total assets)",
    "Attr27": "profit on operating activities / financial expenses",
    "Attr28": "working capital / fixed assets",
    "Attr29": "log(total assets)",
    "Attr30": "total liabilities / cash",
    "Attr31": "(short-term liabilities × 365) / COGS",
    "Attr32": "operating expenses / short-term liabilities",
    "Attr33": "operating expenses / total liabilities",
    "Attr34": "profit on sales / total assets",
    "Attr35": "total sales / total assets (alt)",
    "Attr36": "total sales / total assets",
    "Attr37": "(current assets - inventory - short-term receivables) / short-term liabilities",
    "Attr38": "total liabilities / total assets (alt)",
    "Attr39": "net profit / total assets (alt)",
    "Attr40": "total assets / total liabilities (alt)",
    "Attr41": "short-term liabilities / total assets",
    "Attr42": "(short-term liabilities + long-term liabilities) / equity",
    "Attr43": "profit on operating activities / depreciation",
    "Attr44": "total liabilities / (profit + depreciation)",
    "Attr45": "profit on sales / sales",
    "Attr46": "(current assets - inventory) / short-term liabilities",
    "Attr47": "(current assets - short-term liabilities) / (current assets - inventory - short-term liabilities)",
    "Attr48": "total liabilities / total assets (alt 2)",
    "Attr49": "(net profit + depreciation) / total liabilities",
    "Attr50": "total expenses / total sales",
    "Attr51": "total liabilities / (sales + other operating income)",
    "Attr52": "short-term liabilities / sales",
    "Attr53": "short-term liabilities / total assets",
    "Attr54": "total liabilities / (net profit + depreciation)",
    "Attr55": "profit on operating activities (×1000) / short-term liabilities",
    "Attr56": "net profit / total assets - ROA",
    "Attr57": "total assets / current liabilities",
    "Attr58": "total costs / total sales",
    "Attr59": "long-term liabilities / equity",
    "Attr60": "sales / inventory",
    "Attr61": "sales / receivables",
    "Attr62": "short-term liabilities × 365 / sales",
    "Attr63": "sales / short-term liabilities",
    "Attr64": "sales / fixed assets",
}

# Named columns already in dataset
NAMED_MAP = {
    "net_profit_to_assets"               : "Attr1  — net profit / total assets",
    "total_liabilities_to_assets"        : "Attr2  — total liabilities / total assets",
    "working_capital_to_assets"          : "Attr3  — working capital / total assets",
    "current_assets_to_short_liabilities": "Attr4  — current assets / short-term liabilities",
    "cash_to_current_liabilities"        : "Attr5  — cash / current liabilities",
    "retained_earnings_to_assets"        : "Attr6  — retained earnings / total assets",
    "ebit_to_assets"                     : "Attr7  — EBIT / total assets",
    "book_value_equity_to_liabilities"   : "Attr8  — book value of equity / total liabilities",
    "sales_to_assets"                    : "Attr9  — sales / total assets",
    "equity_to_assets"                   : "Attr10 — equity / total assets",
}


# ── Step 1: Print definition of each candidate variable ───────────────────────

print("\n" + "=" * 70)
print("CANDIDATE VARIABLE DEFINITIONS")
print("=" * 70)

rows = []
for feat in candidates.index:
    if feat in NAMED_MAP:
        definition = NAMED_MAP[feat]
    elif feat in ATTR_MAP:
        definition = ATTR_MAP[feat]
    else:
        definition = "— unknown —"
    rows.append({"feature": feat, "definition": definition})
    print(f"  {feat:<35s} {definition}")

def_df = pd.DataFrame(rows).set_index("feature")


# ── Step 2: Correlation-based duplicate detection ─────────────────────────────
# Variables with |r| > 0.98 are treated as near-duplicates.
# Among each duplicate pair, retain the one with higher |rank-biserial r|.

print("\n" + "=" * 70)
print("CORRELATION-BASED DUPLICATE DETECTION  (|r| > 0.98)")
print("=" * 70)

cand_cols = candidates.index.tolist()
corr_mat  = cohort[cand_cols].corr().abs()

duplicate_pairs = []
checked = set()
for i, c1 in enumerate(cand_cols):
    for j, c2 in enumerate(cand_cols):
        if j <= i:
            continue
        r = corr_mat.loc[c1, c2]
        if r > 0.98:
            duplicate_pairs.append((c1, c2, round(r, 4)))
            print(f"  |r|={r:.4f}  {c1}  ↔  {c2}")
            print(f"         {ATTR_MAP.get(c1, NAMED_MAP.get(c1, '?'))}")
            print(f"         {ATTR_MAP.get(c2, NAMED_MAP.get(c2, '?'))}")

if not duplicate_pairs:
    print("  No near-duplicate pairs found (|r| > 0.98).")


# ── Step 3: Select one representative per duplicate group ─────────────────────
# Keep the variable with higher |rank-biserial r| (stronger univariate signal).

to_drop = set()
for c1, c2, r in duplicate_pairs:
    rb1 = abs(candidates.loc[c1, "rank_biserial"])
    rb2 = abs(candidates.loc[c2, "rank_biserial"])
    drop = c2 if rb1 >= rb2 else c1
    keep = c1 if rb1 >= rb2 else c2
    to_drop.add(drop)
    print(f"\n  Keep: {keep} (|rb|={max(rb1,rb2):.4f})  →  Drop: {drop} (|rb|={min(rb1,rb2):.4f})")

final_candidates = candidates.drop(index=list(to_drop))

print(f"\nVariables dropped as duplicates : {len(to_drop)}")
print(f"Final candidate list            : {len(final_candidates)}")


# ── Step 4: Final variable list with definitions ──────────────────────────────

print("\n" + "=" * 70)
print("FINAL VARIABLE LIST FOR NB04")
print("=" * 70)

final_df = def_df.loc[final_candidates.index].copy()
final_df["OR_mle"]        = candidates.loc[final_candidates.index, "OR_mle"]
final_df["rank_biserial"] = candidates.loc[final_candidates.index, "rank_biserial"]
final_df["direction"]     = candidates.loc[final_candidates.index, "direction"]

for feat, row in final_df.iterrows():
    print(f"  {feat:<35s} OR={row['OR_mle']:.4f}  rb={row['rank_biserial']:.4f}"
          f"  [{row['direction']}]")
    print(f"    → {row['definition']}")

out_path = TABLE_DIR + "NB03b_variable_map.csv"
final_df.to_csv(out_path)
print(f"\nSaved: {out_path}")
print("\nNext step → NB04_association_rules.ipynb")

Candidate variables from NB03 : 16
['working_capital_to_assets', 'Attr25', 'Attr24', 'Attr30', 'Attr12', 'Attr55', 'Attr41', 'retained_earnings_to_assets', 'Attr56', 'Attr29', 'Attr45', 'Attr21', 'cash_to_current_liabilities', 'Attr58', 'Attr57', 'Attr47']

CANDIDATE VARIABLE DEFINITIONS
  working_capital_to_assets           Attr3  — working capital / total assets
  Attr25                              equity / total assets (alt)
  Attr24                              gross profit (3-year avg) / total assets
  Attr30                              total liabilities / cash
  Attr12                              gross profit / short-term liabilities
  Attr55                              profit on operating activities (×1000) / short-term liabilities
  Attr41                              short-term liabilities / total assets
  retained_earnings_to_assets         Attr6  — retained earnings / total assets
  Attr56                              net profit / total assets - ROA
  Attr29             

In [5]:
lr_df = pd.read_csv("../results/tables/NB03_01_univariate_logistic.csv", index_col=0)
print(lr_df[["OR_mle", "OR_firth", "p_firth_fdr"]].head(10))

                             OR_mle  OR_firth   p_firth_fdr
feature                                                    
Attr26                       0.6286    0.6300  7.055245e-12
Attr16                       0.6304    0.6320  2.772116e-11
total_liabilities_to_assets  1.3683    1.3686  0.000000e+00
Attr51                       1.3658    1.3661  0.000000e+00
Attr35                       0.6461    0.6461  0.000000e+00
Attr22                       0.6879    0.6878  0.000000e+00
Attr14                       0.6944    0.6943  0.000000e+00
ebit_to_assets               0.6944    0.6943  0.000000e+00
Attr18                       0.6944    0.6943  0.000000e+00
net_profit_to_assets         0.7016    0.7015  0.000000e+00
